# Generacion Programada de Objetos 3D

Este notebook muestra como crear geometria 3D de forma flexible a partir de listas de coordenadas usando `numpy`, `trimesh`, `open3d` y `vedo`.

Objetivos:
- Definir puntos en el espacio 3D con datos estructurados.
- Generar cubos, esferas y cilindros mediante bucles y condicionales.
- Variar tamano, color y forma segun reglas programadas.
- Exportar resultados en `.OBJ`, `.STL` y `.GLTF/.GLB`.

In [ ]:
# Si usas Colab, descomenta esta linea:
# %pip install -q numpy trimesh open3d vedo

import os
from pathlib import Path
import numpy as np
import trimesh
import open3d as o3d
from vedo import Mesh, show, write

np.random.seed(7)

ModuleNotFoundError: No module named 'open3d'

In [2]:
# 1) Crear una lista de coordenadas 3D a partir de datos estructurados

grid_x = np.linspace(-3.0, 3.0, 5)
grid_y = np.linspace(-3.0, 3.0, 5)
coords = []

for i, x in enumerate(grid_x):
    for j, y in enumerate(grid_y):
        z = 0.35 * np.sin(x) + 0.35 * np.cos(y)
        coords.append([x, y, z])

coords = np.array(coords, dtype=float)
print(f'Total de puntos generados: {len(coords)}')
print('Primeros 5 puntos:\n', coords[:5])

Total de puntos generados: 25
Primeros 5 puntos:
 [[-3.         -3.         -0.39588938]
 [-3.         -1.5        -0.02463398]
 [-3.          0.          0.300608  ]
 [-3.          1.5        -0.02463398]
 [-3.          3.         -0.39588938]]


In [3]:
# 2) Generar primitivas (cubo, esfera, cilindro) desde los puntos
#    usando bucles y condicionales para variar parametros

meshes_tm = []            # mallas trimesh
meshes_vedo = []          # mallas vedo
meta_info = []            # informacion auxiliar por objeto

for idx, (x, y, z) in enumerate(coords):
    # Condicional 1: forma segun indice
    if idx % 3 == 0:
        primitive_type = 'box'
        size = 0.25 + 0.05 * abs(z)
        m = trimesh.creation.box(extents=[size, size, size])
    elif idx % 3 == 1:
        primitive_type = 'sphere'
        radius = 0.18 + 0.04 * abs(z)
        m = trimesh.creation.icosphere(subdivisions=2, radius=radius)
    else:
        primitive_type = 'cylinder'
        radius = 0.10 + 0.03 * abs(z)
        height = 0.35 + 0.10 * (1.0 + z)
        m = trimesh.creation.cylinder(radius=radius, height=max(0.15, height), sections=24)

    # Posicionar el objeto en su coordenada
    m.apply_translation([x, y, z])

    # Condicional 2: color segun altura z
    if z > 0.25:
        color = np.array([230, 80, 70, 220], dtype=np.uint8)   # rojizo
    elif z < -0.25:
        color = np.array([60, 120, 230, 220], dtype=np.uint8)  # azulado
    else:
        color = np.array([80, 200, 130, 220], dtype=np.uint8)  # verdoso

    m.visual.vertex_colors = np.tile(color, (len(m.vertices), 1))

    meshes_tm.append(m)
    meshes_vedo.append(Mesh([m.vertices, m.faces]).c(color[:3] / 255.0))
    meta_info.append({'index': idx, 'type': primitive_type, 'center': [float(x), float(y), float(z)]})

print(f'Objetos 3D creados: {len(meshes_tm)}')
print('Ejemplo de metadata del primer objeto:', meta_info[0])

NameError: name 'Mesh' is not defined

In [ ]:
# 3) Visualizacion rapida de la escena con vedo
# Nota: en algunos entornos remotos puede abrir ventana externa.

_ = show(meshes_vedo, axes=1, bg='white', viewup='z', title='Escena Parametrica 3D', interactive=False)

In [ ]:
# 4) Preparar carpeta de salida y malla combinada

out_dir = Path('exports_3d')
out_dir.mkdir(parents=True, exist_ok=True)

combined_tm = trimesh.util.concatenate(meshes_tm)
print('Vertices combinados:', len(combined_tm.vertices))
print('Caras combinadas:', len(combined_tm.faces))

In [ ]:
# 5) Exportar con trimesh.exchange.export.export_mesh()

obj_path = out_dir / 'escena_trimesh.obj'
stl_path = out_dir / 'escena_trimesh.stl'

trimesh.exchange.export.export_mesh(combined_tm, obj_path)
trimesh.exchange.export.export_mesh(combined_tm, stl_path)

# GLTF/GLB con trimesh (escena completa)
scene = trimesh.Scene(meshes_tm)
glb_path = out_dir / 'escena_trimesh.glb'
scene.export(glb_path)

print('Exportado con trimesh:')
print(' -', obj_path)
print(' -', stl_path)
print(' -', glb_path)

In [ ]:
# 6) Exportar con vedo.write()

vedo_mesh = Mesh([combined_tm.vertices, combined_tm.faces])
vedo_obj = out_dir / 'escena_vedo.obj'
vedo_stl = out_dir / 'escena_vedo.stl'

write(vedo_mesh, str(vedo_obj))
write(vedo_mesh, str(vedo_stl))

print('Exportado con vedo.write():')
print(' -', vedo_obj)
print(' -', vedo_stl)

In [ ]:
# 7) Exportar con open3d.io.write_triangle_mesh()

o3d_mesh = o3d.geometry.TriangleMesh()
o3d_mesh.vertices = o3d.utility.Vector3dVector(combined_tm.vertices)
o3d_mesh.triangles = o3d.utility.Vector3iVector(combined_tm.faces)
o3d_mesh.compute_vertex_normals()

o3d_obj = out_dir / 'escena_open3d.obj'
o3d_stl = out_dir / 'escena_open3d.stl'

open3d_ok_obj = o3d.io.write_triangle_mesh(str(o3d_obj), o3d_mesh)
open3d_ok_stl = o3d.io.write_triangle_mesh(str(o3d_stl), o3d_mesh)

print('Exportado con open3d.io.write_triangle_mesh():')
print(' -', o3d_obj, 'OK=' + str(open3d_ok_obj))
print(' -', o3d_stl, 'OK=' + str(open3d_ok_stl))

In [ ]:
# 8) Verificacion final de archivos exportados

for p in sorted(out_dir.glob('*')):
    size_kb = p.stat().st_size / 1024
    print(f'{p.name:24s} -> {size_kb:8.2f} KB')